day06

0. 복습
함수 매핑

1) Series.map({값 : 바꿀값})
2) Series.apply(함수)
3) DataFrame.apply(axis=0/1)
4) DataFrame.map()

그룹 연산

1) df.groupby("기준열")["대상열"].집계함수()
2) df.groupby(["기준열1", "기준열2"])["대상열"].집계함수()
3) df.groupby("기준열")["대상열"].agg([함수1, 함수2,...])
4) df.groupby("기준열").agg({"열:함수", "열:함수",...})

1. 피벗 테이블 
 : 데이터를 한 열은 행 방향, 다른 한 열은 열 뱡향으로 펼치고
   그 교차 칸에 집계 값을 채운 데이터
- "종" X "성별", "요일" X "시간대" 두 가지 기준이 만나는
  지점의 값을 한눈에 비교할 수 있다

1) pivot_table()
- 두 기준을 행과 열로 펼치기
- "무엇을(values) 어떻게(aggfunc) 집계해서, 행(index)과
   열(columns)로 펼칠까"
- 옵션
	index : 행으로 넣을 열이름
	columns : 열로 넣을 열이름
	values : 집계할 값 열이름
	aggfunc : 어떻게 집계할지(기본 "mean")

2) pd.crosstab()
- 개수 세기 전용
- "각 조합에 몇개 있나"같은 개수 세기만 할거라면,
  crosstab()이 편하다
- value, aggfunc 없이 열만 넣으면 교차 빈도표를 만든다

3) 피벗의 반대 - melt()
- 피벗이 긴 데이터 -> 넓은 표로 펼치는 것이라면,
  반대로 넓은 표 -> 긴 표로 녹여 세운다
- 여러 열에 흩어진 값을 한 열로 모을 때 쓴다

2. 문자열 데이터 처리
 : 데이터의 문자열 값들을 정리하고, 쪼개고, 원하는 부분을 뽑아내는 것
- 문자열은 그대로 두면 문제가 생기는 경우가 있다
ex) "VIP"와 "	VIP"는 사람 눈엔 같지만, 컴퓨터에겐
     공백하나 때문에 다른 값이 된다
    "gold"와 "Gold"도 컴퓨터에게는 완전히 다른 데이터다	

1) .str
- 파이썬에서 문자열 메소드를 쓸때 문자열에 바로 사용
  ex) "hello".upper()
- 판다스는 데이터가 여러개 있기 때문에 바로 사용 불가
- 그래서 판다스는 .str이라는 걸 사용한다
- 열.str.메소드()
  => 해당 열의 모든 문자열에 메소드를 한번에 적용

2) 공백, 대소문자 정리
 : 파이썬의 문자열 메소드와 동일
메소드		기능
=========================================================
.str.stript()	앞뒤 공백 제거
.str.lstrip()	왼쪽 공백 제거 
.str.rstrip()	오른쪽 공백 제거
.str.lower()	모두 소문자로
.str.upper()	모두 대문자로
.str.title()	단어 첫 글자만 대문자로 "gold ring" ->
					"Gold Ring"
.str.capitalize() 문장 첫글자만 대문자로 "gold ring" ->
					 "Gold ring"

3) .str.replace()
- 문자열 안의 특정 문자열을 다른 문자열로 바꾼다
  .str.repalce("바꿀대상", "새값")

4) .str.split()
 : 한 칸에 여러 정보가 뭉쳐 있을때, 구분자 기준으로 잘라 나눈다
- str.split("구분자") 구분자로 잘라 리스트로 만든다
- .str.split("구분자", expend=True)
			잘린 조각을 여러 열로 펼친다
- .str.get(인덱스)	리스트에서 인덱스 위치의 문자열을 꺼냄
- .str[시작:끝:간격]	슬라이싱

5) str.contains()
- 특정 문자열이 있는 행만 골라낼때 사용
- df[조건]에 넣으면 원하는 행만 가져온다(필터링)
- str.contains("문자열") : 해당 문자열이 있으면 True
- str.startswith("문자열") : 해당 문자열이 시작하면 True
- str.endswith("문자열") : 해당 문자열이 끝나면 True

6) zfill()
 : 정리한 값들을 다시 하나로 합치거나, 자릿수를 맞춰 정리할 때 사용 
- 문자열 연결 : 문자열 + 문자열 (시퀀스 연산자)
- .str.zfill(자리수) : 앞을 0으로 채워 자릿수를 맞춘다
	ex) "7" -> "0007"

In [ ]:
## 피벗 테이블
import pandas as pd
import seaborn as sns

# 펭귄 데이터 불러오기
penguins = sns.load_dataset("penguins")

penguins.head()
# groupby로 종 x 성별 평균 몸무게 확인
penguins.groupby(['species', 'sex'], observed=True)['body_mass_g'].mean()
# pivot_table()로 종(행) x 성별(열) -> 평균 몸무게(값)

result = pd.pivot_table(
    penguins,
    index = "species", # 행 -> 종
    columns= "sex", # 열 -> 성별
    values= "body_mass_g", # 값 -> 몸무게
    aggfunc="mean" # 집계 : 평균(기본값이라 생략 가능)
)

result.round(2)
# 종 x 성별 데이터 개수 확인
pd.pivot_table(
    penguins,
    index = "species", # 행 -> 종
    columns= "sex", # 열 -> 성별
    values="body_mass_g",
    aggfunc="count"
)
# 여러 집계를 한번에 보고 싶다면
# aggfunc=[함수1, 함수2,...]

# 평균 값과 최대값을 동시에
pd.pivot_table(
    penguins,
    index = "species", # 행 -> 종
    columns= "sex", # 열 -> 성별
    values="body_mass_g",
    aggfunc=["mean", 'max']
)
# +) margin = True를 주면 맨 아래와 맨 오른쪽에 전체 집계(All)이 붙는다
# => 행별, 열별 전체 경향을 함께 확인하기 좋다
pd.pivot_table(
    penguins,
    index = "species", # 행 -> 종
    columns= "sex", # 열 -> 성별
    values="body_mass_g",
    aggfunc='mean',
    margins=True # 총계(All)추가
)
# 맨 오른쪽 All열은 종별 전체 평균,
# 맨 아래 All행은 성별 전체 평균
# 오른쪽 아래 끝은 전체 펭귄의 평균
# 두 기준 중 데이터가 없는(결측치, NaN) 칸은 NaN이 된다.
# 해결) fill_value = NaN 대체값

# ex) 종마다 사는 섬이 정해져 있어, '섬 x 종'조합에는 NaN 빈칸이 생긴다
# 섬(행) x 종(열) -> 평균 몸무게 => 없는 조합은 NaN이 된다
pd.pivot_table(
    penguins,
    index = "island", # 행 -> 종
    columns= "species", # 열 -> 성별
    values="body_mass_g",
    aggfunc='mean',
    fill_value=0 # NaN을 0으로
)
### crosstab()
# 종(행) x 섬(열)로 몇 마리씩 있는지 세기
pd.crosstab(penguins['species'], penguins['island'])
# 각 종이 어느 섬에 몇마리 사는지 한눈에 보인다.
# +) normalize로 개수 대신 비율을 볼 수 있다.
# normalize='index는 행(종) 기준 비율
(pd.crosstab(penguins['species'], penguins['island'], normalize='index')) * 100
# index='columns'는 서식지 기준 비율
(pd.crosstab(penguins['species'], penguins['island'], normalize='columns')) * 100
# index='True'는 전체 기준 비율
(pd.crosstab(penguins['species'], penguins['island'], normalize=True)) * 100
### melt()
# 과목이 '열'로 넓게 펴진 데이터
wide = pd.DataFrame({
    "이름" : ["펭수", '뽀로로'],
    "국어" : [90, 80],
    '영어' : [85, 95]
})
wide
# melt : '이름'은 고정하고, 국어 및 영어 열을 '과목'-점수
# 두 열로 녹여 세운다
# id_vars : 고정할 열
# var_name : 녹인 열 이름들이 담길 새 열 이름
# value_name : 그 값들이 담길 새 열 이름
pd.melt(wide, id_vars="이름", var_name='과목', value_name='점수')
# <피벗 실습>
import pandas as pd
import seaborn as sns

# 팁 데이터 불러오기
tips = sns.load_dataset('tips')

tips.head()
# 1) 요일(day) x 시간대(time)를 각각 행과 열로, 팁(tip)의 평균을 원소로
#    가지는 피벗테이블 생성
pd.pivot_table(
    tips,
    index='day',
    columns='time',
    values='tip',
    aggfunc='mean',
    observed=True
)
# 토, 일은 점심 데이터가 없어 NaN이다
# 2) 요일(day) x 성별(sex)로 결재액(total_bill)의 평균을 구하면서,
#    margins=True로 전체 평균 확인
pd.pivot_table(
    tips,
    index='day',
    columns='sex',
    values='total_bill',
    aggfunc='mean',
    observed=True,
    margins=True
)
# 3) crosstab()으로 요일(day) x 시간대(time)의 손님 팀 수를 보기
pd.crosstab(tips['day'], tips['time'])
## 문자열 데이터 처리
import pandas as pd

member = pd.read_csv("./회원명단.csv")
member
# 각 이름의 문자열 길이 확인 -> 공백도 한 글자로 count
member["이름"].str.len()
# 공백이 있는지 눈으로 확인
print(f"[{member['이름'][0]}]")
print(f"[{member['이름'][1]}]")
# 이름 앞뒤 공백 제거
member['이름'] = member["이름"].str.strip()
print("--- 결과 확인 ---")
print(member['이름'].str.len())
# 등급은 공백 제거와 대문자 변환을 이어서 처리한다
# .str 두번 연달아 사용
print(member['등급'].value_counts())
member['등급'] = member['등급'].str.strip().str.upper()
member
print(member['등급'].value_counts())
# 전화번호의 하이픈(-)을 없애 숫자만 남기기
member['전화번호'] = member['전화번호'].str.replace("-", "")
member
# '@'를 기준으로 이메일 자르기 -> 행마다 [아이디, 도메인] 리스트
member["이메일"].str.split("@")
# expand = True : 쪼갠 조각을 열로 펼칠 수 있다
result = member['이메일'].str.split("@", expand=True)
result
# 이메일을 '@'로 쪼갠 다음 도메인만 골라 새로운 열로 생성
member['도메인'] = member['이메일'].str.split("@").str.get(1)
member
# 가입일의 앞 4글자 연도만 잘라내기
member['가입년도'] = member['가입일'].str[:4]
member
# 이메일에 'gmail'이 들어간 회원만 필터링
member['이메일'].str.contains('gmail')
# 이메일에 'gmail'이 포함되어 있으면 True
# 필터링 사용
result = member[member['이메일'].str.contains("gmail")]
result
# 이름과 등급을 합치기(문자열 연결)
result = member['이름'].str.strip() + "(" + member['등급'] + ")"
result
# 회원번호 생성
# 회원마다 M0001 같은 회원 번호를 만들기
# 1 ~ 8 번호 -> astype(str)로 문자열 "1" ~ "8" 형변환
member_id = pd.Series(range(1, 9)).astype(str)

# zfill(4) : 앞을 0으로 채워 4자리로 ('1' -> '0001'), 그 앞에 "M"을 붙임
member['회원번호'] = 'M' + member_id.str.zfill(4)
member

### 과제

## day06 과제

### 피벗테이블

seaborn `titanic` 데이터로 피벗테이블을 연습합니다. 아래 셀을 먼저 실행하세요.

- 주요 열 : `class`(객실등급 First/Second/Third), `sex`(성별), `survived`(생존 1/0), `fare`(요금)

In [ ]:
import pandas as pd
import seaborn as sns

titanic = sns.load_dataset("titanic")
print(titanic[["class", "sex", "survived", "fare"]].head())

문제 1) pivot_table 기본 — 등급 × 성별 생존율

- **객실등급(`class`)을 행, 성별(`sex`)을 열**로 놓고, **생존율(`survived`의 평균)**을 값으로 하는 피벗테이블을 소수 셋째 자리까지 만드세요. (힌트 : `pd.pivot_table(index=, columns=, values=, aggfunc="mean")`)

<출력결과>

sex     female   male
class
First    0.968  0.369
Second   0.921  0.157
Third    0.500  0.135

In [ ]:
result = pd.pivot_table(
    titanic,
    index="class",
    columns="sex",
    values="survived",
    aggfunc="mean",
    observed=True
)
result.round(3)

문제 2) margins — 등급 × 성별 평균 요금 + 전체

- **등급 × 성별**로 **평균 요금(`fare`)**을 소수 둘째 자리까지 구하되, `margins=True`로 **전체 평균(All)** 행·열도 함께 보이게 하세요.

<출력결과>

sex     female   male    All
class
First   106.13  67.23  84.15
Second   21.97  19.74  20.66
Third    16.12  12.66  13.68
All      44.48  25.52  32.20

In [ ]:
result = pd.pivot_table(
    titanic,
    index="class",
    columns="sex",
    values="fare",
    aggfunc="mean",
    observed=True,
    margins=True
)
result.round(2)

문제 3) crosstab — 등급 × 성별 인원수

- `pd.crosstab`으로 **등급(`class`) × 성별(`sex`)** 조합별 **인원수**를 세어 출력하세요.

<출력결과>

sex     female  male
class
First       94   122
Second      76   108
Third      144   347

In [ ]:
pd.crosstab(titanic["class"], titanic["sex"])

---

### 문자열 데이터 처리

일부러 지저분하게 만든 **`day06_고객주문.csv`**(주문 고객 120명) 파일로 문자열 처리를 연습합니다. 이 CSV는 과제 노트북과 **같은 폴더**에 있습니다. 아래 셀을 먼저 실행하세요.

- `고객명` : 앞뒤에 공백이 섞여 있음
- `이메일` : 대소문자가 뒤섞여 있고 도메인이 다양함(gmail/naver/daum/company/kakao)
- `주문코드` : `분류약자-번호` 형태 (EL 전자 / FD 식품 / BK 도서 / HM 생활 / SP 스포츠)
- `등급` : `gold`/`Gold`/`GOLD`처럼 대소문자·공백이 제각각
- `결제액` : 결제 금액(숫자)

> 💡 데이터가 120행이라, 표를 통째로 보는 문제는 `head()`(앞부분)로 확인합니다.

In [ ]:
# 지저분한 고객 주문 데이터 불러오기 (한글이 있어 utf-8-sig 로 읽는다)
order = pd.read_csv("day06_고객주문.csv", encoding="utf-8-sig")
print(order.head())

문제 4) 공백 제거 + 대소문자 통일 — strip / upper

- `등급` 열의 **앞뒤 공백을 없애고, 모두 대문자로 통일**해 원래 열에 저장하세요. (힌트 : `.str.strip().str.upper()`)
- 정리한 뒤 `value_counts()`로 등급별 인원을 세어 출력하세요. (정리 안 하면 `gold`/`Gold`/`GOLD`가 따로 세어집니다)

<출력결과>

등급
SILVER    42
VIP       39
GOLD      39
Name: count, dtype: int64

In [ ]:
order["등급"] = order["등급"].str.strip().str.upper()
result = order["등급"].value_counts()
result

문제 5) 쪼개서 뽑기 — split + get

- `주문코드`(`EL-1024` 형태)를 `-` 기준으로 쪼개, **분류 약자(EL/FD/BK/HM/SP)** 만 뽑아 새 열 `분류` 로 저장하세요. (힌트 : `.str.split("-").str.get(0)`)
- `주문코드`·`분류`를 **앞 5줄** 출력하고, `분류`별 개수도 세어 출력하세요.

<출력결과>

주문코드  분류
0  FD-3286  FD
1  EL-2535  EL
2  HM-4611  HM
3  BK-5552  BK
4  BK-5333  BK
분류
SP    30
BK    26
EL    24
FD    21
HM    19
Name: count, dtype: int64

In [ ]:
order["분류"] = order["주문코드"].str.split("-").str.get(0)
print(order[["주문코드", "분류"]].head())
print(order["분류"].value_counts())

문제 6) 검색·필터 — contains

- `이메일`에 **`gmail`이 들어간 고객만** 걸러내세요. (힌트 : `df[df["이메일"].str.contains("gmail")]`)
- **gmail 고객이 몇 명**인지 출력하고(`shape[0]`), `고객명`·`이메일`을 **앞 5줄** 출력하세요.

<출력결과>

gmail 고객 수: 24
     고객명                이메일
1    권가은   doyoon@gmail.com
6    최유진   doyoon@gmail.com
13   윤준서    yeeun@gmail.com
21   최도윤  seoyeon@gmail.com
29  장서연      dave@gmail.com

In [ ]:
result = order[order["이메일"].str.contains("gmail")]
print(f"gmail 고객 수 : {len(result)}")
result[["고객명", "이메일"]].head()

문제 7) 자릿수 채우기 — zfill로 회원번호 만들기

- 1부터 순번을 매겨, **`M0001` 형태의 4자리 회원번호**를 만들어 새 열 `회원번호` 로 저장하세요.
    - 힌트 : `pd.Series(range(1, len(order)+1)).astype(str)` 로 순번 문자열을 만들고, `.str.zfill(4)` 로 4자리를 맞춘 뒤 앞에 `"M"` 을 붙입니다.
- `고객명`·`회원번호`를 **앞 5줄** 출력하세요.

<출력결과>

고객명   회원번호
0  한지우  M0001
1  권가은  M0002
2  김가은  M0003
3  신민준  M0004
4  강지우  M0005

In [ ]:
order_id = pd.Series(range(1, len(order)+1)).astype(str)
order["회원번호"] = "M" + order_id.str.zfill(4)
order[['고객명', '회원번호']].head()